# hparam-precedence-merge — ex1: merge defaults < config-file < CLI args via dict.update

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `hparam-precedence-merge`. Running the final beacon cell reports progress against the `Config: hparam precedence merge` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: hparam precedence merge` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`hparam-precedence-merge`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "hparam-precedence-merge"
DD_SUBTOPIC = "Config: hparam precedence merge"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: hparam precedence merge — quick refresher

Real training scripts pull config from multiple sources with a clear precedence chain. The standard order (lowest → highest):

```
defaults  <  config file  <  CLI overrides  <  programmatic
```

Implementation is just chained `dict.update()`:

```python
def merge_config(defaults, file_cfg, cli_args):
    out = {}
    out.update(defaults)
    out.update(file_cfg)   # file overrides defaults
    out.update(cli_args)   # CLI overrides file
    return out
```

**Why later-wins is the right semantic.** The user typed the CLI args most recently — they're the strongest expression of intent. A YAML config they wrote yesterday is a weaker signal. Defaults are the weakest signal of all (we picked them).

**`dict.update(other)` is shallow.** If a value is a nested dict (e.g. `{'optimizer': {'lr': 1e-3}}`), `update` REPLACES the whole subdict instead of merging into it. For nested configs you want a recursive merge (Hydra, OmegaConf). But for flat training args the shallow merge is correct.

**Don't mutate the inputs.** Start with `out = {}` so the caller's `defaults` dict isn't polluted across runs.

### Exercise 1 — merge defaults < config-file < CLI args via dict.update

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply chained `dict.update` to merge three config layers (defaults, file, CLI) with strict later-overrides-earlier semantics, without mutating any input.
> Keywords: config, precedence, merge, cli
> ```

**KCs targeted:** `dict-update-later-wins`, `input-isolation-no-mutation`

Implement `ex1_merge_config(defaults, file_cfg, cli_args)`. The standard training-script config merge.

Precedence (lowest → highest):
  `defaults  <  file_cfg  <  cli_args`

So CLI overrides file overrides defaults. Same as argparse + YAML.

Algorithm:
1. Start with an empty dict `out = {}`.
2. `out.update(defaults)` (lowest precedence — fills every default key).
3. `out.update(file_cfg)` (file overrides).
4. `out.update(cli_args)` (CLI overrides — wins).
5. Return `out`.

Constraints:
- Must NOT mutate any of the three input dicts.
- A `None` value in `cli_args` should STILL override (it's an explicit choice). Don't filter Nones — that's a different design.
- A key that appears ONLY in `file_cfg` (not in defaults) is allowed — pass it through.

Output: merged `dict`.

In [ ]:
def ex1_merge_config(defaults: dict, file_cfg: dict, cli_args: dict) -> dict:
    """Merge three config layers; later overrides earlier."""
    raise NotImplementedError()


def _test_ex1():
    # === Basic precedence chain ===
    defaults = {'lr': 1e-3, 'batch_size': 32, 'epochs': 10}
    file_cfg = {'lr': 3e-4, 'epochs': 5}
    cli_args = {'lr': 1e-4}

    merged = ex1_merge_config(defaults, file_cfg, cli_args)
    assert merged == {'lr': 1e-4, 'batch_size': 32, 'epochs': 5}, (
        f'precedence wrong: {merged}\n'
        f'expected lr from CLI, batch_size from defaults, epochs from file'
    )

    # === Inputs are NOT mutated ===
    assert defaults == {'lr': 1e-3, 'batch_size': 32, 'epochs': 10}, 'defaults was mutated'
    assert file_cfg == {'lr': 3e-4, 'epochs': 5}, 'file_cfg was mutated'
    assert cli_args == {'lr': 1e-4}, 'cli_args was mutated'

    # === All three layers empty ===
    assert ex1_merge_config({}, {}, {}) == {}

    # === Only defaults ===
    assert ex1_merge_config({'a': 1}, {}, {}) == {'a': 1}

    # === Only file ===
    assert ex1_merge_config({}, {'a': 1}, {}) == {'a': 1}

    # === CLI introduces a new key ===
    assert ex1_merge_config({'a': 1}, {}, {'b': 2}) == {'a': 1, 'b': 2}

    # === None in CLI overrides ===
    out = ex1_merge_config({'lr': 1e-3}, {}, {'lr': None})
    assert out == {'lr': None}, f'None should override, got {out}'

    # === File-only key passes through ===
    out = ex1_merge_config({'a': 1}, {'b': 2}, {})
    assert out == {'a': 1, 'b': 2}

    # === Result is a fresh dict (not an alias) ===
    d = {'x': 1}
    out = ex1_merge_config(d, {}, {})
    out['y'] = 99
    assert 'y' not in d, 'mutating the result must not back-propagate to defaults'

    # === Order matters — three-way tie test ===
    # All three layers set 'lr'. CLI must win.
    out = ex1_merge_config({'lr': 1.0}, {'lr': 2.0}, {'lr': 3.0})
    assert out['lr'] == 3.0, f'CLI must win three-way; got {out}'
    # Without CLI, file wins.
    out = ex1_merge_config({'lr': 1.0}, {'lr': 2.0}, {})
    assert out['lr'] == 2.0
    # Without file, defaults.
    out = ex1_merge_config({'lr': 1.0}, {}, {})
    assert out['lr'] == 1.0
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_merge_config(defaults, file_cfg, cli_args):
    out = {}
    out.update(defaults)
    out.update(file_cfg)
    out.update(cli_args)
    return out
```

**`dict.update(other)` is the right primitive.** It writes every key from `other` into `self`, OVERWRITING existing values. That's exactly the later-wins semantic.

**Alternative one-liner: `{**defaults, **file_cfg, **cli_args}`.** Same semantics; sometimes considered more Pythonic. The explicit `update` form scales better if you add a fourth layer (e.g. env vars) — just one more line.

**Don't filter out `None` from `cli_args`.** It's tempting to write `{k: v for k, v in cli_args.items() if v is not None}` to support `--lr` (no value = 'use default'). But that conflates 'not provided' with 'explicit None'. argparse handles this at the parser level by only adding a key if it was provided — use that mechanism, not post-hoc None filtering.

**Shallow merge limitation.** If `defaults['optimizer'] = {'lr': 1e-3, 'beta1': 0.9}` and `file_cfg['optimizer'] = {'lr': 3e-4}`, the merge gives `{'optimizer': {'lr': 3e-4}}` — `beta1` is lost. Deep configs need OmegaConf or Hydra; flat configs don't.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()